# 01 · Prepare the synthetic transfer-request extract

**Question:** can the raw export be turned into a clean, de-duplicated cohort with a well-defined target and a defensible train/validation/test split?

All data is synthetic. No real patient, facility, or payer information is used anywhere.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

# Anchor on this project specifically: it sits in a subdirectory of a repository
# that has its own pyproject.toml, so "nearest pyproject.toml" is not enough.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "transfer_decline").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


## The raw file, warts and all
The extract imitates a transfer-center report pulled from Epic. It carries the messiness such a pull usually has: inconsistent capitalisation and stray whitespace in `transport_mode` and `referring_facility_type`, missing values in a few columns, and some double-entered requests.

In [2]:
from transfer_decline.data import load_raw
raw = load_raw(ROOT)
print(raw.shape)
print('exact duplicate rows:', raw.duplicated().sum())
display(raw['transport_mode'].value_counts())
display(raw['referring_facility_type'].value_counts())
display(raw.isna().sum()[lambda s: s > 0])

(9060, 22)
exact duplicate rows: 60


transport_mode
Ground ALS                         7586
Referring Facility Transport        833
  Ground ALS                        344
Rotor Wing                          247
  Referring Facility Transport       40
  Rotor Wing                         10
Name: count, dtype: int64

referring_facility_type
Community Hospital ED           3784
Community Hospital Inpatient    2171
Critical Access Hospital        1121
Skilled Nursing Facility         860
Physician Office                 588
COMMUNITY HOSPITAL ED            249
COMMUNITY HOSPITAL INPATIENT     131
CRITICAL ACCESS HOSPITAL          66
SKILLED NURSING FACILITY          53
PHYSICIAN OFFICE                  37
Name: count, dtype: int64

bed_assignment_datetime    1904
arrival_datetime           1904
distance_miles              175
payer                       263
acuity_score                141
decline_reason             7597
inpatient_los_days         1904
icu_upgrade_within_24h     1904
dtype: int64

## Cleaning
`clean()` drops the exact duplicates, trims and case-normalises the two text columns, derives the binary target, and adds calendar features from the request timestamp. Missing values are **left in place** — imputation belongs in the modelling pipeline where it is fit on the training split only.

**Target definition:** `declined = 1` when `disposition == 'Declined'`. `Accepted - Not Transferred` counts as an acceptance: the transfer center said yes even though the patient never arrived.

In [3]:
from transfer_decline.data import clean, time_split, validate
df = clean(raw)
df['split'] = time_split(df)
print('rows after clean:', len(df))
display(df['disposition'].value_counts())
print('overall decline rate: {:.1%}'.format(df['declined'].mean()))
display(df.groupby('split')['declined'].agg(n='size', rate='mean'))

rows after clean: 9000


disposition
Accepted - Transferred        7114
Declined                      1453
Accepted - Not Transferred     433
Name: count, dtype: int64

overall decline rate: 16.1%


,n,rate
split,,
test,1800,0.187222
train,5400,0.174259
validation,1800,0.097222


## Why a chronological split
The model is meant to score a request as it arrives, using only history. A time-ordered split — earliest 60% train, next 20% validation, last 20% test — matches that and stops near-identical requests from the same shift landing in two partitions. The validation split is what the operating threshold is tuned on in notebook 05; the test split is never touched until the final evaluation.

In [4]:
checks = validate(df)
print(json.dumps(checks, indent=2, default=str))
assert checks['no_exact_duplicates']
assert checks['target_is_binary']
assert checks['splits_are_time_ordered']

{
  "rows": 9000,
  "no_exact_duplicates": true,
  "target_is_binary": true,
  "decline_rate": 0.1614,
  "facility_type_no_nulls_after_norm": true,
  "transport_mode_values": [
    "Ground ALS",
    "Referring Facility Transport",
    "Rotor Wing"
  ],
  "split_counts": {
    "train": 5400,
    "validation": 1800,
    "test": 1800
  },
  "splits_are_time_ordered": true
}


In [5]:
prepared_path = ROOT / 'data' / 'synthetic' / 'prepared_transfer_requests.csv'
df.to_csv(prepared_path, index=False)
print('wrote', prepared_path.relative_to(ROOT))

wrote data\synthetic\prepared_transfer_requests.csv


## What this establishes
A reproducible cohort with a clear target and an honest split. It does **not** establish clinical or operational realism — the decline rate, the missingness pattern, and the payer mix are all properties of the synthetic generator, not findings about any real transfer center.